Summary
=======

In this notebook I want to derive the algorithm for constructing a PPM (*Prediction by Partial Match*) model for text content. This is based on

*"On Prediction Using Variable Order Markov Models"* by Ron Begleiter, Ran El-Yaniv, and Golan Yona in
the Journal of Artificial Intelligence Research 22 (2004) 385-421.

I have already implemented the code for this on github [here](https://github.com/rpgomez/vomm) but that was for the scenario in which the size of the alphabet is relative small ($\lessapprox 256$), not the case where the size of the alphabet is large ($\gtrapprox 2^{15}$) such as the vocabulary of a large collection of documents. 

In the second case, the amount of memory required to build a PPM model is significantly higher than necessary with a simple, naive approach. So I'm going to document here my approach for conserving on memory in order to implement a reasonable PPM model where the vocabulary are words and a sequence of symbols are a sequence of grammatically correct words.

Lastly, I would like to be able to compare the performance of the various models on a corpus of Herman Melville's Moby Dick.


## Estimating the probability distribution for a categorical variable $X$ with sparse data.

* I have a catecorical variable $X$ which takes integer values $1,2, \ldots, N$ whenre $N \gg 1$
* I have a set of i.i.d. drawn samples of $X$, $s_1,s_2,\ldots, s_M$ where $M \ll N$
* I want to estimate the probability distribution $Pr(X=i)$ from my samples.

What is the appropriate algorithm?

### Solution: Simplify the problem

I have a random variable $Y$ with 2 possible values $0,1$ with observed counts of $n_0, n_1$. The maximum likelihood estimator (MLE) for $Pr(Y=i)$ is given by 

$$Pr(Y=i) = \frac{n_i}{n_0 + n_1}$$

To estimate the uncertainty of my estimate I can use Bayesian inference with a [Dirichlet](https://en.wikipedia.org/wiki/Dirichlet_distribution) [Conjugate Prior](https://en.wikipedia.org/wiki/Conjugate_prior) to estimate $Pr(Y=i)$. I start by using the uninformative prior $Dir(\alpha= (1,1))$ and then I use the observed counts to update my belief to a posterior distribution:
$Dir(\alpha = (1 + n_0,1 + n_1))$. 

Using my posterior distribution I can now determine the most likely maximum a posteriori (MAP) distribution (the same as MLE in this case), but can also measure the variance of the possible distributions that explain the observed data.

### The more complicated problem
I have a categorical value $X$ with $N \gg 1$, but I only I get to make at most $M \ll N$ observations. How can I use the simplified problem to estimate $Pr(X=i)$?

**Solution**:

1. Without loss of generality, let $1,\ldots,R$ be the observed values in the sequence $s_1,\ldots, s_M$.
2. Let $Y$ denote a random variable which takes on values in the range $1,\ldots, R$.
3. Let $Z$ denote a random variable which takes on values in the range $R+1,\ldots, N$.
4. We can interpret $X$ a mixture of the 2 random variables $Y,Z$ as follows:
\begin{align}
Pr(X = i) &= Pr(X=i, Y \text{ is selected}) + Pr(X=i, Z\text{ is selected}) \\
&= Pr(X=i| Y \text{ is selected}) Pr(Y \text{ is selected}) + Pr(X=i| Z\text{ is selected})Pr( Z\text{ is selected}) \\
&= Pr(X=i| Y \text{ is selected}) \omega + Pr(X=i| Z\text{ is selected})(1-\omega) \\
&= Pr(Y=i| Y \text{ is selected}) \omega + Pr(Z=i| Z\text{ is selected})(1-\omega) \\
&= Pr(Y=i) \omega + Pr(Z=i)(1-\omega) \\
\end{align}
where $0 \le \omega \le 1$ is the mixing parameter.

We can estimate $\omega$ using the simplified problem:

$$(\omega, 1-\omega) \sim Dir\left(\alpha = (1 + n_Y, 1 + n_Z)\right) = Dir\left(\alpha = (1 + M, 1)\right)$$

whose mean value is given $ \omega = (1 + M)/(2 + M)$. We should not use the MAP/MLE value:
$\omega = M/M = 1$ as that implies observations of $Z$ never occur.



We can estimate the probability distribution for $Y$ using the simplified problem.

Let $c_i, i = 1, \ldots, R$ be the observed frequency counts of $X=i$. Then $$Y \sim Dir(\alpha = ( 1+c_1, \ldots, 1 + c_R))$$

We can choose to use the mean value distribution for $Y$:

$$Pr(Y=i) = \frac{1 + c_i}{R + \sum_j c_j}$$

or the MLE/MAP:

$$Pr(Y=i) = \frac{c_i}{\sum_j c_j}$$


Because we never get to observe $Z$ we only have the uninformative prior $Z \sim Dir(\alpha = (1,1,\ldots,1))$ whose mean distribution for $Z$ is $Pr(Z = i) = 1/(N - R)$. Because $1-\omega \ll \omega$ using the mean distribution should not cause significant errors in predictions.

### Estimating $Pr(word| context)$

From above we can now estimate for a given observed context of size $d$ the probability distribution of the word that follows. 

In order to conserve memory 

* we first translate all words into numbers, $1, 2 , \ldots, W$ where $W$ is the size of the vocabulary
* create a dictionary *Counts* with keys all observed contexts of size $d$ and values another dictionary where
    * the value is a dictionary that has (key, value) pairs (word, # of occurrences after the specific context), i.e.
     Counts(c)(w) = # of times word *w* was observed to follow context *c*
* create a dictionary *Probabilities* 
    * with keys all observed contexts of size $d$ 
    * and for each key a dictionary with (w, p) pair where 
        * *w* is a word observed to follow the context 
        * and p is the probability of *w* occurring given by 
        $$\omega_{c}\times c_w/\sum_v c_v$$ where 
            * $\omega_c$ is the mixing factor determined for the given context $c$, 
            * and $c_v$ is the number of times word $v$ was observed to follow context $c$.
        * There is 1 additional (w,p) pair with key  *none* and value $(1-\omega_c)/(W-R_c)$ where
            * $W-R_c$ is the number of vocabulary words that were **not** observed to follow context $c$.
        

     



In [ ]:
import numpy as np
from collections import Counter
from collections import defaultdict
from tqdm.notebook import tqdm

import vomm

# Moby Dick
I'm going to create a PPM model for the content of Moby Dick.

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize
import nltk.tokenize
import string

In [ ]:
nltk.download('punkt')

In [ ]:
# Moby Dick
!wget -c https://www.gutenberg.org/cache/epub/2701/pg2701.txt

In [ ]:
content = open('pg2701.txt','r').read().lower()

In [ ]:
filter = string.ascii_letters
def filter_nonletters(aletter):
 if aletter in filter:
    return aletter
 else:
    return " "
new_content = "".join([filter_nonletters(x) for x in tqdm(content)])

In [ ]:
words = nltk.tokenize.word_tokenize(new_content)
print("# of words in the content: ",len(words))

vocabulary = set(words)
print("Size of vocabulary: ", len(vocabulary))

numbers_to_words = list(vocabulary)
words_to_numbers = {}
for t,word in enumerate(numbers_to_words):
    words_to_numbers[word] = t

obs_sequence = tuple([words_to_numbers[word] for word in words])

In [ ]:
letters = [x for x in new_content]
print("# of letters in the content: ",len(letters))

vocabulary_letters = set(letters)
print("Size of vocabulary: ", len(vocabulary_letters))

numbers_to_letters = list(vocabulary_letters)
letters_to_numbers = {}
for t,word in enumerate(numbers_to_letters):
    letters_to_numbers[word] = t

obs_sequence_letters = tuple([letters_to_numbers[word] for word in letters])

In [ ]:
%%time
my_model_ppm_words = vomm.PPM_words()
my_model_ppm_words.fit(obs_sequence,verbose=True)

In [ ]:
%%time
score = my_model_ppm_words.logpdf(generated_sequence)
print("log probability of observed sequence: ",score)

## Letter models
I want to models on letter tokens instead of word tokens.

In [ ]:
%%time
my_model_ppm_letters = vomm.ppm()
my_model_ppm_letters.fit(obs_sequence_letters,d=6)

In [ ]:
%%time
my_model_pst_letters = vomm.pst()
my_model_pst_letters.fit(obs_sequence_letters,d=6)

In [ ]:
%%time
my_model_pst_js_letters = vomm.pst_JS()
my_model_pst_js_letters.fit(obs_sequence_letters,d=6,js_threshold=0.02)

## Comparing the models against the original source

In [ ]:
for a_model in [my_model_ppm_letters, my_model_pst_letters, my_model_pst_js_letters]:
    score = a_model.logpdf(obs_sequence_letters[:1000])
    print("My model's stats: ")
    print(a_model)
    print("Log probability for the first 1000 source tokens: ", score)
    print()

## Generated Content.
I want to see how each of the 3 models generates new data.

In [ ]:
%%time
generated_sequence = my_model_ppm_letters.generate_data(length=200)
generated_words = "".join([numbers_to_letters[x] for x in generated_sequence])
print("generated phrase:\n",generated_words)

In [ ]:
%%time
generated_sequence = my_model_pst_letters.generate_data(length=200)
generated_words = "".join([numbers_to_letters[x] for x in generated_sequence])
print("generated phrase:\n",generated_words)

In [ ]:
%%time
generated_sequence = my_model_pst_js_letters.generate_data(length=200)
generated_words = "".join([numbers_to_letters[x] for x in generated_sequence])
print("generated phrase:\n",generated_words)

## Model sizes
Let's see how large each type of model is.

In [ ]:
print(my_model_ppm_letters)
print()
print(my_model_pst_letters)
print()
print(my_model_pst_js_letters)
print()
print(my_model_ppm_words)